## Registering COGs as Earth Engine Assets

Created by Mel Rose; Last updated 9/12/2026

## Setup

In [12]:
#TODO: This script is behind the current model version (1.0.6) Change to cn paths after merging to updated model branch.
#TODO: Add in other variations (by gas, land state node, land use, SOC gain, SOC loss, organic soil state, combined LULUCF, etc)
#TODO: Require which gas to run from command line argument, if none supplied default to all_gases. Update in both s3 and gcs dirs.

#TODO: Update with cn patterns
#TODO: Update with the additional datasets, years, make stacked bands, remove test
veg_model_version_underscore = "1_0_5"
ee_project = 'landandcarbon'
gcs_bucket = 'lcl_public'
veg_asset_folder = f'wri_lgms/vegetation/{veg_model_version_underscore}'
year = 2017

# GCS directories and filenames
emissions_directory = f'{veg_asset_folder}/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e'
removals_directory = f'{veg_asset_folder}/removals/gross_removals__all_C_pools__MgCO2'
emissions_filename = f'gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_{year}.tif'
removals_filename = f'gross_removals__all_C_pools__MgCO2_ha_yr_{year}.tif'

# GCP asset folder
emissions_asset_folder = f'{veg_asset_folder}/emissions'
removals_asset_folder = f'{veg_asset_folder}/removals'

In [10]:
from google.cloud import storage
import ee
import json
from pprint import pprint
import google.auth
from google.auth.transport.requests import AuthorizedSession

credentials, _ = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/cloud-platform'
    ]
)

credentials = credentials.with_quota_project(ee_project)
ee.Initialize(credentials=credentials, project=ee_project)
session = AuthorizedSession(credentials)

In [13]:
# Verify that COGs exist in the GCS bucket
storage_client = storage.Client(project=ee_project)
bucket = storage_client.get_bucket(gcs_bucket)

# TODO: Update with the additional datasets
cogs = {
    'emissions': f'{emissions_directory}/{emissions_filename}',
    'removals': f'{removals_directory}/{removals_filename}',
}

for dataset, object_name in cogs.items():
    blob = bucket.blob(object_name)
    print(f'{dataset}: gs://{gcs_bucket}/{object_name} -> exists={blob.exists()}')

emissions: gs://lcl_public/wri_lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017.tif -> exists=True
removals: gs://lcl_public/wri_lgms/vegetation/v1_0_5/removals/gross_removals__all_C_pools__MgCO2/gross_removals__all_C_pools__MgCO2_ha_yr_2017.tif -> exists=True


## Register the COGs in GEE

Each GeoTIFF is registered as its own COG-backed `ee.Image` asset. The source remains in Google Cloud Storage; Earth Engine references the external COG rather than ingesting a copy.

Information on image manifests and registering COGs as assets:
* https://developers.google.com/earth-engine/guides/image_manifest
* https://developers.google.com/earth-engine/Earth_Engine_asset_from_cloud_geotiff

In [6]:
# Create an Earth Engine folder hierarchy if it does not already exist
def ensure_ee_folder(folder_path, ee_project):


    parts = folder_path.strip('/').split('/')
    current_path = f'projects/{ee_project}/assets'

    for part in parts:
        current_path = f'{current_path}/{part}'

        try:
            ee.data.getAsset(current_path)
            print(f'Exists:  {current_path}')

        except ee.EEException:
            ee.data.createAsset(
                {'type': 'FOLDER'},
                current_path
            )
            print(f'Created: {current_path}')

# Function to execute COG registration request
def register_cog_as_ee_asset(session, request, ee_project):
  url = f'https://earthengine.googleapis.com/v1alpha/projects/{ee_project}/image:importExternal'

  response = session.post(
    url = url,
    data = json.dumps(request)
  )

  return json.loads(response.content)

In [14]:
# TODO: Update with the additional datasets
# TODO: Make multiband images for timeseries per dataset
# TODO: Remove "test" from asset_name

# Create the GEE folder structure if it doesn't already exist
ensure_ee_folder(emissions_asset_folder, ee_project)
ensure_ee_folder(removals_asset_folder, ee_project)

# Define the COG-backed image manifests
asset_specs = {
    'emissions': {
        'asset_name': f'gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_{year}__test',
        'asset_folder': emissions_asset_folder,
        'gcs_uri': f'gs://{gcs_bucket}/{emissions_directory}/{emissions_filename}',
    },
    'removals': {
        'asset_name': f'gross_removals__all_C_pools__MgCO2_ha_yr_{year}__test',
        'asset_folder': removals_asset_folder,
        'gcs_uri': f'gs://{gcs_bucket}/{removals_directory}/{removals_filename}',
    },
}

requests = {}
for dataset, spec in asset_specs.items():
    requests[dataset] = {
        'imageManifest': {
            'name': f"projects/{ee_project}/assets/{spec['asset_folder']}/{spec['asset_name']}",
            'tilesets': [
                {
                    'sources': [
                        {'uris': [spec['gcs_uri']]}
                    ]
                }
            ],
            'startTime': f'{year}-01-01T00:00:00.000000000Z',
            'endTime': f'{year + 1}-01-01T00:00:00.000000000Z',
            'properties': {
                'year': year,
                'dataset': dataset,
                'version': 'v1.0.5',
            },
        }
    }

pprint(requests)

{'emissions': {'imageManifest': {'endTime': '2018-01-01T00:00:00.000000000Z',
                                 'name': 'projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017__test',
                                 'properties': {'dataset': 'emissions',
                                                'version': 'v1.0.5',
                                                'year': 2017},
                                 'startTime': '2017-01-01T00:00:00.000000000Z',
                                 'tilesets': [{'sources': [{'uris': ['gs://lcl_public/wri_lgms/vegetation/v1_0_5/emissions/gross_emissions__all_C_pools__all_gases__MgCO2e/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017.tif']}]}]}},
 'removals': {'imageManifest': {'endTime': '2018-01-01T00:00:00.000000000Z',
                                'name': 'projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_removals__all_C_pools__MgCO2_ha_yr_2017__test',


In [16]:
# Execute both COG registration requests
results = {}
for dataset, request in requests.items():
    results[dataset] = register_cog_as_ee_asset(session, request, ee_project)
    print(f'\n{dataset}:')
    pprint(results[dataset])


emissions:
{}

removals:
{}


## Verify the registered assets


In [17]:
# Print the expected Earth Engine asset IDs
asset_ids = {
    dataset: f"projects/{ee_project}/assets/{spec['asset_folder']}/{spec['asset_name']}"
    for dataset, spec in asset_specs.items()
}

pprint(asset_ids)

{'emissions': 'projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr_2017__test',
 'removals': 'projects/landandcarbon/assets/wri_lgms/vegetation/v1_0_5/gross_removals__all_C_pools__MgCO2_ha_yr_2017__test'}


In [ ]:
# Load the registered assets in Earth Engine and inspect metadata
for dataset, asset_id in asset_ids.items():
    print(f'\n{dataset}: {asset_id}')
    pprint(ee.Image(asset_id).getInfo())

{}